In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, LogNorm, Normalize
import seaborn as sns
import numpy as np
from collections import defaultdict

In [2]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
FANTOM_INITIAL = DATA_INITIAL / "fantom_cat"
PREPROCESSING = Path("../preprocessing")

MATRICES = PREPROCESSING / "matrices"
IPS_MATRICES = MATRICES / "ips_matrices"
FIBRO_MATRICES = MATRICES / "fibro_matrices"

INFO = PREPROCESSING / "info"
IPS_INFO = INFO / "ips_info"
FIBRO_INFO = INFO / "fibro_info"

FISHER = PREPROCESSING / "fisher_test"
IPS_FISHER = FISHER / "ips_fisher_exact_test"
FIBRO_FISHER = FISHER / "fibro_fisher_exact_test"

RESULTS = Path("../results")
FISHER_RESULTS = RESULTS / "fisher_test"
PERMUTATION_RES_AFTER_FDR = RESULTS / "rnachrom_himorna-intersection"

In [3]:
fisher_results_filt_by_exp = {}
for exp in ["fibro", "ips"]:
    to_file = FISHER_RESULTS / f"{exp}-fisher-significant.tsv"
    fisher_results_filt = pd.read_csv(to_file, sep="\t")
    if exp == "fibro":
        fisher_results_filt["aso"] = fisher_results_filt["aso"].apply(lambda x: "_".join(x.split("_")[1:]))
    fisher_results_filt["lnc-hm"] = fisher_results_filt["lnc"] + "-" + fisher_results_filt["hm"]
    fisher_results_filt["lnc-aso-hm"] = fisher_results_filt["lnc"] + "-" + fisher_results_filt["aso"] + "-" + fisher_results_filt["hm"]
    fisher_results_filt_by_exp[exp] = fisher_results_filt    

In [4]:
fisher_results_filt_by_exp["fibro"]

,hm,lnc_ens,aso,er,ea,wa,wr,lnc,lnc-hm,lnc-aso-hm
0,H3K9me3,ENSG00000163364,G0163364_AD_01,1.000000e+00,1.917683e-03,1.000000,1.000000,AC017048.3,AC017048.3-H3K9me3,AC017048.3-G0163364_AD_01-H3K9me3
1,H3K9me3,ENSG00000214548,G0214548_01,6.904775e-01,6.172479e-04,0.001780,0.696110,MEG3,MEG3-H3K9me3,MEG3-G0214548_01-H3K9me3
2,H3K9me3,ENSG00000214548,G0214548_AD_09,1.000000e+00,1.050761e-07,1.000000,0.000695,MEG3,MEG3-H3K9me3,MEG3-G0214548_AD_09-H3K9me3
3,H3K9me3,ENSG00000223478,G0223478_03,8.484336e-04,9.853195e-01,1.000000,1.000000,RP11-545E17.3,RP11-545E17.3-H3K9me3,RP11-545E17.3-G0223478_03-H3K9me3
4,H3K9me3,ENSG00000225470,G0225470_AD_04,1.000000e+00,1.000000e+00,0.002088,1.000000,JPX,JPX-H3K9me3,JPX-G0225470_AD_04-H3K9me3
...,...,...,...,...,...,...,...,...,...,...
277,H3K4me1,ENSG00000268895,G0268895_01,1.000000e+00,4.303351e-01,0.005312,1.000000,A1BG-AS1,A1BG-AS1-H3K4me1,A1BG-AS1-G0268895_01-H3K4me1
278,H3K4me1,ENSG00000268895,G0268895_03,7.864094e-04,4.142816e-01,0.000205,1.000000,A1BG-AS1,A1BG-AS1-H3K4me1,A1BG-AS1-G0268895_03-H3K4me1
279,H3K4me1,ENSG00000269834,G0269834_02,6.742146e-03,5.148661e-01,0.255875,1.000000,CTD-3018O17.3,CTD-3018O17.3-H3K4me1,CTD-3018O17.3-G0269834_02-H3K4me1
280,H3K4me1,ENSG00000269834,G0269834_AD_02,1.579246e-07,3.774202e-01,0.443793,1.000000,CTD-3018O17.3,CTD-3018O17.3-H3K4me1,CTD-3018O17.3-G0269834_AD_02-H3K4me1


In [8]:
to_file = PERMUTATION_RES_AFTER_FDR / "permutation_fdr_concat-signif.tsv"
all_perm_res_adj_signif = pd.read_csv(to_file, sep="\t")
all_perm_res_adj_signif.head(3)

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant,lnc
0,ENSG00000130600,H3K27ac,0.009901,HFFc6,0.017327,True,H19
1,ENSG00000130600,H3K27ac,0.009901,iMARGI-HFFc6,0.017327,True,H19
2,ENSG00000130600,H3K27ac,0.009901,iMARGI-HEK293T,0.017327,True,H19


In [9]:
all_perm_res_adj_signif.shape

(9368, 7)

In [13]:
all_perm_res_adj_signif_multiindex = all_perm_res_adj_signif.set_index(["hm", "lnc_ens"])
all_perm_res_adj_signif_multiindex

p_value                  rnachrom_expr  \
hm          lnc_ens                                                    
H3K27ac     ENSG00000130600  0.009901                          HFFc6   
            ENSG00000130600  0.009901                   iMARGI-HFFc6   
            ENSG00000130600  0.009901                 iMARGI-HEK293T   
            ENSG00000130600  0.009901                   iMARGI-HUVEC   
            ENSG00000130600  0.019802                     GRID-MM_1S   
...                               ...                            ...   
methylation ENSG00000281398  0.009901                     Red_C-K562   
            ENSG00000282057  0.009901                          HFFc6   
            ENSG00000282057  0.009901                   iMARGI-HFFc6   
            ENSG00000282057  0.009901  GRID-MDA_MB_231__HTB_26_ATCC_   
            ENSG00000282057  0.009901              Red_C-fibroblasts   

                                p_adj  significant  lnc  
hm          lnc_ens                                      
H3K27ac     ENSG00000130600  0.017327         True  H19  
            ENSG00000130600  0.017327         True  H19  
            ENSG00000130600  0.017327         True  H19  
            ENSG00000130600  0.017327         True  H19  
            ENSG00000130600  0.027723         True  H19  
...                               ...          ...  ...  
methylation ENSG00000281398  0.023102         True  NaN  
            ENSG00000282057  0.009901         True  NaN  
            ENSG00000282057  0.009901         True  NaN  
            ENSG00000282057  0.009901         True  NaN  
            ENSG00000282057  0.009901         True  NaN  

[9368 rows x 5 columns]

In [14]:
fisher_results_filt_by_exp_multiindex = fisher_results_filt_by_exp["fibro"].set_index(["hm", "lnc_ens"])
fisher_results_filt_by_exp_multiindex

aso            er            ea        wa  \
hm      lnc_ens                                                                 
H3K9me3 ENSG00000163364  G0163364_AD_01  1.000000e+00  1.917683e-03  1.000000   
        ENSG00000214548     G0214548_01  6.904775e-01  6.172479e-04  0.001780   
        ENSG00000214548  G0214548_AD_09  1.000000e+00  1.050761e-07  1.000000   
        ENSG00000223478     G0223478_03  8.484336e-04  9.853195e-01  1.000000   
        ENSG00000225470  G0225470_AD_04  1.000000e+00  1.000000e+00  0.002088   
...                                 ...           ...           ...       ...   
H3K4me1 ENSG00000268895     G0268895_01  1.000000e+00  4.303351e-01  0.005312   
        ENSG00000268895     G0268895_03  7.864094e-04  4.142816e-01  0.000205   
        ENSG00000269834     G0269834_02  6.742146e-03  5.148661e-01  0.255875   
        ENSG00000269834  G0269834_AD_02  1.579246e-07  3.774202e-01  0.443793   
        ENSG00000271270     G0271270_09  8.601842e-08  1.000000e+00  1.000000   

                               wr            lnc                 lnc-hm  \
hm      lnc_ens                                                           
H3K9me3 ENSG00000163364  1.000000     AC017048.3     AC017048.3-H3K9me3   
        ENSG00000214548  0.696110           MEG3           MEG3-H3K9me3   
        ENSG00000214548  0.000695           MEG3           MEG3-H3K9me3   
        ENSG00000223478  1.000000  RP11-545E17.3  RP11-545E17.3-H3K9me3   
        ENSG00000225470  1.000000            JPX            JPX-H3K9me3   
...                           ...            ...                    ...   
H3K4me1 ENSG00000268895  1.000000       A1BG-AS1       A1BG-AS1-H3K4me1   
        ENSG00000268895  1.000000       A1BG-AS1       A1BG-AS1-H3K4me1   
        ENSG00000269834  1.000000  CTD-3018O17.3  CTD-3018O17.3-H3K4me1   
        ENSG00000269834  1.000000  CTD-3018O17.3  CTD-3018O17.3-H3K4me1   
        ENSG00000271270  1.000000      TMCC1-AS1      TMCC1-AS1-H3K4me1   

                                                   lnc-aso-hm  
hm      lnc_ens                                                
H3K9me3 ENSG00000163364     AC017048.3-G0163364_AD_01-H3K9me3  
        ENSG00000214548              MEG3-G0214548_01-H3K9me3  
        ENSG00000214548           MEG3-G0214548_AD_09-H3K9me3  
        ENSG00000223478     RP11-545E17.3-G0223478_03-H3K9me3  
        ENSG00000225470            JPX-G0225470_AD_04-H3K9me3  
...                                                       ...  
H3K4me1 ENSG00000268895          A1BG-AS1-G0268895_01-H3K4me1  
        ENSG00000268895          A1BG-AS1-G0268895_03-H3K4me1  
        ENSG00000269834     CTD-3018O17.3-G0269834_02-H3K4me1  
        ENSG00000269834  CTD-3018O17.3-G0269834_AD_02-H3K4me1  
        ENSG00000271270         TMCC1-AS1-G0271270_09-H3K4me1  

[282 rows x 8 columns]

In [33]:
merged_results = pd.merge(left=fisher_results_filt_by_exp_multiindex, right=all_perm_res_adj_signif_multiindex, how="inner", left_index=True, right_index=True)
merged_results

aso   er   ea        wa        wr  \
hm      lnc_ens                                                         
H3K9me3 ENSG00000225470  G0225470_AD_04  1.0  1.0  0.002088  1.000000   
        ENSG00000225470  G0225470_AD_04  1.0  1.0  0.002088  1.000000   
        ENSG00000225470  G0225470_AD_04  1.0  1.0  0.002088  1.000000   
        ENSG00000225470  G0225470_AD_04  1.0  1.0  0.002088  1.000000   
        ENSG00000225470  G0225470_AD_04  1.0  1.0  0.002088  1.000000   
...                                 ...  ...  ...       ...       ...   
H3K4me1 ENSG00000257219  G0257219_AD_05  1.0  1.0  0.000026  0.014333   
        ENSG00000257219  G0257219_AD_05  1.0  1.0  0.000026  0.014333   
        ENSG00000257219  G0257219_AD_05  1.0  1.0  0.000026  0.014333   
        ENSG00000257219  G0257219_AD_05  1.0  1.0  0.000026  0.014333   
        ENSG00000257219  G0257219_AD_05  1.0  1.0  0.000026  0.014333   

                               lnc_x               lnc-hm  \
hm      lnc_ens                                             
H3K9me3 ENSG00000225470          JPX          JPX-H3K9me3   
        ENSG00000225470          JPX          JPX-H3K9me3   
        ENSG00000225470          JPX          JPX-H3K9me3   
        ENSG00000225470          JPX          JPX-H3K9me3   
        ENSG00000225470          JPX          JPX-H3K9me3   
...                              ...                  ...   
H3K4me1 ENSG00000257219  RP11-54A9.1  RP11-54A9.1-H3K4me1   
        ENSG00000257219  RP11-54A9.1  RP11-54A9.1-H3K4me1   
        ENSG00000257219  RP11-54A9.1  RP11-54A9.1-H3K4me1   
        ENSG00000257219  RP11-54A9.1  RP11-54A9.1-H3K4me1   
        ENSG00000257219  RP11-54A9.1  RP11-54A9.1-H3K4me1   

                                                 lnc-aso-hm   p_value  \
hm      lnc_ens                                                         
H3K9me3 ENSG00000225470          JPX-G0225470_AD_04-H3K9me3  0.009901   
        ENSG00000225470          JPX-G0225470_AD_04-H3K9me3  0.009901   
        ENSG00000225470          JPX-G0225470_AD_04-H3K9me3  0.009901   
        ENSG00000225470          JPX-G0225470_AD_04-H3K9me3  0.009901   
        ENSG00000225470          JPX-G0225470_AD_04-H3K9me3  0.009901   
...                                                     ...       ...   
H3K4me1 ENSG00000257219  RP11-54A9.1-G0257219_AD_05-H3K4me1  0.009901   
        ENSG00000257219  RP11-54A9.1-G0257219_AD_05-H3K4me1  0.009901   
        ENSG00000257219  RP11-54A9.1-G0257219_AD_05-H3K4me1  0.009901   
        ENSG00000257219  RP11-54A9.1-G0257219_AD_05-H3K4me1  0.009901   
        ENSG00000257219  RP11-54A9.1-G0257219_AD_05-H3K4me1  0.009901   

                                         rnachrom_expr     p_adj  significant  \
hm      lnc_ens                                                                 
H3K9me3 ENSG00000225470                          HFFc6  0.011315         True   
        ENSG00000225470                   iMARGI-HFFc6  0.011315         True   
        ENSG00000225470  GRID-MDA_MB_231__HTB_26_ATCC_  0.011315         True   
        ENSG00000225470                 iMARGI-HEK293T  0.011315         True   
        ENSG00000225470                   iMARGI-HUVEC  0.011315         True   
...                                                ...       ...          ...   
H3K4me1 ENSG00000257219                          HFFc6  0.011881         True   
        ENSG00000257219                   iMARGI-HFFc6  0.011881         True   
        ENSG00000257219  GRID-MDA_MB_231__HTB_26_ATCC_  0.011881         True   
        ENSG00000257219                 iMARGI-HEK293T  0.011881         True   
        ENSG00000257219                   iMARGI-HUVEC  0.011881         True   

                               lnc_y  
hm      lnc_ens                       
H3K9me3 ENSG00000225470          JPX  
        ENSG00000225470          JPX  
        ENSG00000225470          JPX  
        ENSG00000225470          JPX  
        ENSG00000225470          JPX  
...            

In [37]:
bootstraped_paires = merged_results.reset_index()[["hm", "lnc_x"]].drop_duplicates()
bootstraped_paires

,hm,lnc_x
0,H3K9me3,JPX
7,H3K9me3,RP11-395B7.4
11,H3K9me3,RP4-773A18.4
16,H3K27me3,RP11-54A9.1
21,H3K9ac,LINC00963
...,...,...
702,H3K4me1,LINC00702
708,H3K4me1,MAGI2-AS3
713,H3K4me1,NR2F1-AS1
721,H3K4me1,RP11-539L10.3


In [39]:
to_save = PERMUTATION_RES_AFTER_FDR / "bootstraped_paires"
bootstraped_paires.to_csv(to_save, sep="\t", index=False)